# Building datasets from measured TESLA cavity data

In [1]:
# --! include root folder into PYTHONPATH --!

import os
import sys

thisdir = os.getcwd()
rootdir = os.path.abspath(os.path.join(thisdir, '..', '..'))
sys.path.append(rootdir)

# --! import python libraries and kind framework --!

import torch
import numpy as np

from matplotlib import pyplot as plt

import utils_data

In [2]:
# --! data readout from a file --!

def convert(s):
    """Converts a string ``s`` to a float while replacing commas with dots."""
    s = s.replace(',', '.')
    return float(s)

def read_detuning(name: str, nrow_skip: int=0, delim: str=None):
    return torch.tensor(
        np.loadtxt(
            name,
            delimiter=delim,
            skiprows=nrow_skip,
            dtype=np.float32,
            ndmin=2,
            converters=convert))

dataname = '../../data/baselines/detuning_17072025_15-10_1_QL_4_5e7_KI_0x10000'
rawdata  = read_detuning(dataname, nrow_skip=23, delim='\t')

print(f'inf >> read data shape is {rawdata.shape}')

inf >> read data shape is torch.Size([1000000, 2])


In [3]:
# --! load a trained model --!

model = torch.load('../../models/baselines/tesla_kind_stat.pt', weights_only=False)
model.eval()

model(
  (operator_stat): operator_stationary(
    (fun_enc): fcnn(
      (net): Sequential(
        (0): Sequential(
          (0): Linear(in_features=40, out_features=128, bias=True)
          (1): ReLU()
        )
        (1): Sequential(
          (0): Linear(in_features=128, out_features=128, bias=True)
          (1): ReLU()
        )
        (2): Sequential(
          (0): Linear(in_features=128, out_features=128, bias=True)
          (1): ReLU()
        )
        (3): Sequential(
          (0): Linear(in_features=128, out_features=320, bias=True)
          (1): Identity()
        )
      )
    )
    (fun_prune): Linear(in_features=8, out_features=8, bias=False)
    (mod_mean): Linear(in_features=8, out_features=8, bias=False)
    (mod_var): Linear(in_features=8, out_features=8, bias=False)
    (pre_mean_dec): fcnn(
      (net): Sequential(
        (0): Sequential(
          (0): Linear(in_features=8, out_features=64, bias=True)
          (1): ReLU()
        )
        (1): Sequen

In [4]:
datum         = rawdata[150_000:150_000+2000, [1]]
dataset_size  = 3200
dataset       = utils_data.create_dataset(dataset_size, model, [datum])

dataset_stat  = [item for item, flag in dataset if flag]
dataset_trans = [item for item, flag in dataset if not flag]

dataset_ratio = len(dataset_trans) / len(dataset) * 100 
print(f'inf >> transient time series make {dataset_ratio:.1f}% of a {dataset_size}-item dataset')

inf >> transient time series make 84.7% of a 3200-item dataset
